# Phase II: Data Preprocessing and Feature Engineering

## 1. Load Original Dataset

## 2. Define Target and Excluded Variables

## 3. Create Stratified Train-Test Split

## 4. Missing and Invalid Value Handling

## 5. Categorical Encoding

## 6. Numerical Transformation

## 7. Class Imbalance Assessment

## 8. Feature Selection

## 9. Export Preprocessed Dataset

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

print("Required libraries imported successfully.")


Required libraries imported successfully.


In [3]:
data_path = Path("../data/telecom_customer_churn.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

df.head()


Dataset shape: (7043, 38)


,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,...,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


In [4]:
df["Churn Target"] = (
    df["Customer Status"] == "Churned"
).astype(int)

excluded_columns = [
    "Customer ID",
    "Customer Status",
    "Churn Category",
    "Churn Reason",
    "Churn Target"
]

X = df.drop(columns=excluded_columns)
y = df["Churn Target"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


Feature shape: (7043, 34)
Target shape: (7043,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

Training features: (5634, 34)
Testing features: (1409, 34)


In [6]:
training_missing = (
    X_train.isna()
    .sum()
    .sort_values(ascending=False)
)

training_missing = training_missing[
    training_missing > 0
]

print(training_missing)

Offer                                3092
Internet Type                        1207
Streaming Music                      1207
Unlimited Data                       1207
Online Security                      1207
Avg Monthly GB Download              1207
Streaming TV                         1207
Premium Tech Support                 1207
Device Protection Plan               1207
Online Backup                        1207
Streaming Movies                     1207
Multiple Lines                        529
Avg Monthly Long Distance Charges     529
dtype: int64


In [7]:
for column in [
    "Internet Type",
    "Streaming TV",
    "Online Security",
    "Offer",
    "Multiple Lines"
]:
    print("\n")
    print(column)
    print(df[column].value_counts(dropna=False))



Internet Type
Internet Type
Fiber Optic    3035
DSL            1652
NaN            1526
Cable           830
Name: count, dtype: int64


Streaming TV
Streaming TV
No     2810
Yes    2707
NaN    1526
Name: count, dtype: int64


Online Security
Online Security
No     3498
Yes    2019
NaN    1526
Name: count, dtype: int64


Offer
Offer
NaN        3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64


Multiple Lines
Multiple Lines
No     3390
Yes    2971
NaN     682
Name: count, dtype: int64


In [8]:
print("Internet Type missing by Internet Service:")
print(
    pd.crosstab(
        df["Internet Service"],
        df["Internet Type"].isna(),
        margins=True
    )
)

print("\nMultiple Lines missing by Phone Service:")
print(
    pd.crosstab(
        df["Phone Service"],
        df["Multiple Lines"].isna(),
        margins=True
    )
)

print("\nOffer missing count:")
print(df["Offer"].isna().sum())

Internet Type missing by Internet Service:
Internet Type     False  True   All
Internet Service                   
No                    0  1526  1526
Yes                5517     0  5517
All                5517  1526  7043

Multiple Lines missing by Phone Service:
Multiple Lines  False  True   All
Phone Service                    
No                  0   682   682
Yes              6361     0  6361
All              6361   682  7043

Offer missing count:
3877


In [9]:
X_train_processed = X_train.copy()
X_test_processed = X_test.copy()

internet_categorical_columns = [
    "Internet Type",
    "Online Security",
    "Online Backup",
    "Device Protection Plan",
    "Premium Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Streaming Music",
    "Unlimited Data"
]

for column in internet_categorical_columns:
    X_train_processed[column] = (
        X_train_processed[column]
        .fillna("No Internet Service")
    )

    X_test_processed[column] = (
        X_test_processed[column]
        .fillna("No Internet Service")
    )

In [10]:
X_train_processed["Multiple Lines"] = (
    X_train_processed["Multiple Lines"]
    .fillna("No Phone Service")
)

X_test_processed["Multiple Lines"] = (
    X_test_processed["Multiple Lines"]
    .fillna("No Phone Service")
)

In [12]:
X_train_processed["Offer"] = (
    X_train_processed["Offer"]
    .fillna("No Offer")
)

X_test_processed["Offer"] = (
    X_test_processed["Offer"]
    .fillna("No Offer")
)

In [13]:
service_numeric_columns = [
    "Avg Monthly GB Download",
    "Avg Monthly Long Distance Charges"
]

for column in service_numeric_columns:
    X_train_processed[column] = (
        X_train_processed[column]
        .fillna(0)
    )

    X_test_processed[column] = (
        X_test_processed[column]
        .fillna(0)
    )

In [14]:
print(
    "Training missing values:",
    int(X_train_processed.isna().sum().sum())
)

print(
    "Testing missing values:",
    int(X_test_processed.isna().sum().sum())
)

print(
    "\nRemaining training columns with missing values:"
)

print(
    X_train_processed.isna()
    .sum()
    .loc[lambda values: values > 0]
)

Training missing values: 0
Testing missing values: 0

Remaining training columns with missing values:
Series([], dtype: int64)


### Missing-Value Treatment

Missing service-related values were treated as structurally missing rather than randomly missing. Missing internet-service categories were represented as "No Internet Service", missing Multiple Lines values were represented as "No Phone Service", and missing Offer values were represented as "No Offer". Missing numerical service-usage values were replaced with zero because the corresponding service was unavailable. The same rules were applied separately to the training and testing datasets.

In [16]:
categorical_columns = (
    X_train_processed
    .select_dtypes(include=["string", "object"])
    .columns
    .tolist()
)

numerical_columns = [
    column
    for column in X_train_processed.columns
    if column not in categorical_columns
]

print("Categorical columns:", len(categorical_columns))
print("Numerical columns:", len(numerical_columns))
print("Total columns:", len(X_train_processed.columns))

Categorical columns: 19
Numerical columns: 15
Total columns: 34


## Invalid Value Investigation

In [17]:
X_train_processed[numerical_columns].describe().T


,count,mean,std,min,25%,50%,75%,max
Age,5634.0,46.549166,16.737607,19.000000,33.000000,46.000000,60.000000,80.000000
Number of Dependents,5634.0,0.471601,0.965401,0.000000,0.000000,0.000000,0.000000,9.000000
Zip Code,5634.0,93479.708378,1851.865197,90001.000000,92104.250000,93516.000000,95322.000000,96150.000000
Latitude,5634.0,36.189215,2.464957,32.555828,33.987945,36.175255,38.135897,41.962127
Longitude,5634.0,-119.752977,2.160784,-124.301372,-121.809955,-119.540290,-117.963696,-114.192901
Number of Referrals,5634.0,1.952964,3.008023,0.000000,0.000000,0.000000,3.000000,11.000000
Tenure in Months,5634.0,32.367412,24.562876,1.000000,9.000000,29.000000,55.000000,72.000000
Avg Monthly Long Distance Charges,5634.0,22.992604,15.399186,0.000000,9.302500,22.950000,36.367500,49.990000
Avg Monthly GB Download,5634.0,20.656550,20.379039,0.000000,4.000000,17.000000,27.000000,85.000000
Monthly Charge,5634.0,63.838090,31.153183,-10.000000,33.787500,70.150000,89.750000,118.750000


In [18]:
for column in numerical_columns:
    negative_count = (
        X_train_processed[column] < 0
    ).sum()

    if negative_count > 0:
        print(column, ":", negative_count)

Longitude : 5634
Monthly Charge : 97


In [19]:
for column in categorical_columns:
    print("\n")
    print(column)
    print(
        X_train_processed[column]
        .value_counts()
        .head(10)
    )



Gender
Gender
Male      2862
Female    2772
Name: count, dtype: int64


Married
Married
No     2936
Yes    2698
Name: count, dtype: int64


City
City
Los Angeles      233
San Diego        226
San Jose          94
San Francisco     83
Sacramento        78
Fresno            46
Escondido         45
Long Beach        44
Oakland           43
Stockton          35
Name: count, dtype: int64


Offer
Offer
No Offer    3092
Offer B      660
Offer E      642
Offer D      478
Offer A      413
Offer C      349
Name: count, dtype: int64


Phone Service
Phone Service
Yes    5105
No      529
Name: count, dtype: int64


Multiple Lines
Multiple Lines
No                  2730
Yes                 2375
No Phone Service     529
Name: count, dtype: int64


Internet Service
Internet Service
Yes    4427
No     1207
Name: count, dtype: int64


Internet Type
Internet Type
Fiber Optic            2442
DSL                    1318
No Internet Service    1207
Cable                   667
Name: count, dtype: int64


O

### Investigation of Negative Monthly Charges

In [20]:
print(
    X_train_processed["Monthly Charge"]
    .describe()
)

print(
    "\nNegative Monthly Charge range:"
)

print(
    X_train_processed.loc[
        X_train_processed["Monthly Charge"] < 0,
        "Monthly Charge"
    ].describe()
)

count    5634.000000
mean       63.838090
std        31.153183
min       -10.000000
25%        33.787500
50%        70.150000
75%        89.750000
max       118.750000
Name: Monthly Charge, dtype: float64

Negative Monthly Charge range:
count    97.000000
mean     -5.484536
std       2.958439
min     -10.000000
25%      -8.000000
50%      -5.000000
75%      -3.000000
max      -1.000000
Name: Monthly Charge, dtype: float64


In [21]:
monthly_charge_check_columns = [
    "Monthly Charge",
    "Total Charges",
    "Total Refunds",
    "Total Revenue",
    "Tenure in Months",
    "Customer Status"
]

available_check_columns = [
    column
    for column in monthly_charge_check_columns
    if column in df.columns
]

negative_monthly_charge_rows = df.loc[
    df["Monthly Charge"] < 0,
    available_check_columns
]

print(
    "Total records with negative Monthly Charge:",
    len(negative_monthly_charge_rows)
)

negative_monthly_charge_rows.head(20)

Total records with negative Monthly Charge: 120


,Monthly Charge,Total Charges,Total Refunds,Total Revenue,Tenure in Months,Customer Status
1,-4.0,542.40,38.33,610.28,9,Stayed
32,-2.0,7942.15,0.00,10830.97,66,Stayed
170,-3.0,465.70,0.00,465.70,7,Stayed
232,-8.0,4539.60,0.00,4669.60,68,Stayed
336,-1.0,343.95,0.00,397.25,5,Churned
428,-10.0,840.10,0.00,840.10,21,Churned
571,-2.0,4138.90,0.00,5867.56,47,Stayed
692,-4.0,24.80,0.00,24.80,1,Churned
694,-7.0,825.70,0.00,1063.08,11,Churned
702,-9.0,4663.40,0.00,7049.00,60,Stayed


In [22]:
negative_charge_summary = (
    negative_monthly_charge_rows[
        [
            "Monthly Charge",
            "Total Charges",
            "Total Refunds",
            "Total Revenue"
        ]
    ]
    .describe()
    .T
)

negative_charge_summary

,count,mean,std,min,25%,50%,75%,max
Monthly Charge,120.0,-5.416667,2.888934,-10.00,-8.000,-5.000,-3.0000,-1.00
Total Charges,120.0,2028.856250,2120.436069,19.40,343.025,1325.975,3239.5125,7942.15
Total Refunds,120.0,3.176667,9.855009,0.00,0.000,0.000,0.0000,49.24
Total Revenue,120.0,2727.278417,2719.182437,22.54,460.800,1920.660,4410.2950,10830.97


In [23]:
negative_train_count = (
    X_train_processed["Monthly Charge"] < 0
).sum()

negative_test_count = (
    X_test_processed["Monthly Charge"] < 0
).sum()

print(
    "Negative Monthly Charge records in training data:",
    negative_train_count
)

print(
    "Negative Monthly Charge records in testing data:",
    negative_test_count
)

print(
    "Total across both partitions:",
    negative_train_count + negative_test_count
)

Negative Monthly Charge records in training data: 97
Negative Monthly Charge records in testing data: 23
Total across both partitions: 120


In [25]:
dictionary_path = Path("../data/telecom_data_dictionary.csv")

data_dictionary = pd.read_csv(
    dictionary_path,
    encoding="cp1252"
)

print("Data dictionary loaded successfully.")
print("Shape:", data_dictionary.shape)

data_dictionary.head()

Data dictionary loaded successfully.
Shape: (40, 3)


,Table,Field,Description
0,Customer Churn,CustomerID,A unique ID that identifies each customer
1,Customer Churn,Gender,"The customer’s gender: Male, Female"
2,Customer Churn,Age,"The customer’s current age, in years, at the t..."
3,Customer Churn,Married,"Indicates if the customer is married: Yes, No"
4,Customer Churn,Number of Dependents,Indicates the number of dependents that live w...


In [26]:
monthly_charge_dictionary = data_dictionary[
    data_dictionary.astype(str)
    .apply(
        lambda row: row.str.contains(
            "Monthly Charge",
            case=False,
            na=False
        ).any(),
        axis=1
    )
]

monthly_charge_dictionary



,Table,Field,Description
29,Customer Churn,Monthly Charge,Indicates the customer’s current total monthly...


In [27]:
monthly_charge_description = (
    monthly_charge_dictionary["Description"]
    .iloc[0]
)

print(monthly_charge_description)

Indicates the customer’s current total monthly charge for all their services from the company


In [28]:
negative_monthly_charge_rows[
    [
        "Monthly Charge",
        "Total Charges",
        "Total Refunds",
        "Total Revenue",
        "Tenure in Months"
    ]
].head(20)


,Monthly Charge,Total Charges,Total Refunds,Total Revenue,Tenure in Months
1,-4.0,542.40,38.33,610.28,9
32,-2.0,7942.15,0.00,10830.97,66
170,-3.0,465.70,0.00,465.70,7
232,-8.0,4539.60,0.00,4669.60,68
336,-1.0,343.95,0.00,397.25,5
428,-10.0,840.10,0.00,840.10,21
571,-2.0,4138.90,0.00,5867.56,47
692,-4.0,24.80,0.00,24.80,1
694,-7.0,825.70,0.00,1063.08,11
702,-9.0,4663.40,0.00,7049.00,60


In [29]:
negative_monthly_charge_rows[
    [
        "Monthly Charge",
        "Total Charges",
        "Total Refunds",
        "Total Revenue",
        "Tenure in Months"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
Monthly Charge,120.0,-5.416667,2.888934,-10.00,-8.000,-5.000,-3.0000,-1.00
Total Charges,120.0,2028.856250,2120.436069,19.40,343.025,1325.975,3239.5125,7942.15
Total Refunds,120.0,3.176667,9.855009,0.00,0.000,0.000,0.0000,49.24
Total Revenue,120.0,2727.278417,2719.182437,22.54,460.800,1920.660,4410.2950,10830.97
Tenure in Months,120.0,29.766667,24.643929,1.00,7.000,22.500,54.0000,72.00


In [30]:
positive_monthly_charge_median = (
    X_train_processed.loc[
        X_train_processed["Monthly Charge"] >= 0,
        "Monthly Charge"
    ]
    .median()
)

print(
    "Training median for valid Monthly Charge values:",
    positive_monthly_charge_median
)

Training median for valid Monthly Charge values: 70.5


In [31]:
negative_train_before = (
    X_train_processed["Monthly Charge"] < 0
).sum()

negative_test_before = (
    X_test_processed["Monthly Charge"] < 0
).sum()

X_train_processed.loc[
    X_train_processed["Monthly Charge"] < 0,
    "Monthly Charge"
] = positive_monthly_charge_median

X_test_processed.loc[
    X_test_processed["Monthly Charge"] < 0,
    "Monthly Charge"
] = positive_monthly_charge_median

print(
    "Training values corrected:",
    negative_train_before
)

print(
    "Testing values corrected:",
    negative_test_before
)

Training values corrected: 97
Testing values corrected: 23


In [32]:
negative_train_after = (
    X_train_processed["Monthly Charge"] < 0
).sum()

negative_test_after = (
    X_test_processed["Monthly Charge"] < 0
).sum()

print(
    "Negative Monthly Charge values remaining in training:",
    negative_train_after
)

print(
    "Negative Monthly Charge values remaining in testing:",
    negative_test_after
)

print(
    "Updated training Monthly Charge range:",
    X_train_processed["Monthly Charge"].min(),
    "to",
    X_train_processed["Monthly Charge"].max()
)


Negative Monthly Charge values remaining in training: 0
Negative Monthly Charge values remaining in testing: 0
Updated training Monthly Charge range: 18.4 to 118.75


## 5. Categorical Encoding

In [33]:
from sklearn.preprocessing import OrdinalEncoder

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train_encoded = X_train_processed.copy()
X_test_encoded = X_test_processed.copy()

X_train_encoded[categorical_columns] = (
    categorical_encoder.fit_transform(
        X_train_processed[categorical_columns]
    )
)

X_test_encoded[categorical_columns] = (
    categorical_encoder.transform(
        X_test_processed[categorical_columns]
    )
)

print("Categorical encoding completed.")

Categorical encoding completed.


In [34]:
remaining_text_train = (
    X_train_encoded
    .select_dtypes(include=["string", "object"])
    .columns
    .tolist()
)

remaining_text_test = (
    X_test_encoded
    .select_dtypes(include=["string", "object"])
    .columns
    .tolist()
)

print(
    "Text columns remaining in training data:",
    remaining_text_train
)

print(
    "Text columns remaining in testing data:",
    remaining_text_test
)

print("Training shape:", X_train_encoded.shape)
print("Testing shape:", X_test_encoded.shape)

Text columns remaining in training data: []
Text columns remaining in testing data: []
Training shape: (5634, 34)
Testing shape: (1409, 34)


### Categorical Encoding Method

The 19 categorical predictor variables were converted into numeric form using ordinal encoding. The encoder was fitted only on the training dataset and then applied to the testing dataset to avoid data leakage. Any category appearing in the testing dataset but not observed during training is represented by -1.

## 7. Class Imbalance Assessment

In [37]:
class_counts = y_train.value_counts().sort_index()

class_percentages = (
    y_train.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("Training class counts:")
print(class_counts)

print("\nTraining class percentages:")
print(class_percentages)

Training class counts:
Churn Target
0    4139
1    1495
Name: count, dtype: int64

Training class percentages:
Churn Target
0    73.46
1    26.54
Name: proportion, dtype: float64


In [38]:
majority_count = class_counts.max()
minority_count = class_counts.min()

imbalance_ratio = majority_count / minority_count

print(
    "Majority-to-minority ratio:",
    round(imbalance_ratio, 2)
)

Majority-to-minority ratio: 2.77


### Class Distribution

The training target distribution was examined to determine whether class imbalance was present. Class 0 represents non-churn customers, while Class 1 represents churned customers. Any imbalance-handling method will be applied only to the training data to prevent information leakage into the test set.

In [39]:
majority_count = class_counts.max()
minority_count = class_counts.min()

imbalance_ratio = majority_count / minority_count

print(
    "Majority-to-minority ratio:",
    round(imbalance_ratio, 2)
)


Majority-to-minority ratio: 2.77


In [40]:
from sklearn.utils import resample

training_combined = X_train_encoded.copy()
training_combined["Churn Target"] = y_train.to_numpy()

majority_training = training_combined[
    training_combined["Churn Target"] == 0
]

minority_training = training_combined[
    training_combined["Churn Target"] == 1
]

minority_oversampled = resample(
    minority_training,
    replace=True,
    n_samples=len(majority_training),
    random_state=42
)

balanced_training = pd.concat(
    [
        majority_training,
        minority_oversampled
    ],
    axis=0
)

balanced_training = balanced_training.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

X_train_balanced = balanced_training.drop(
    columns=["Churn Target"]
)

y_train_balanced = balanced_training[
    "Churn Target"
]

print("Balanced training feature shape:", X_train_balanced.shape)

print("\nBalanced training class counts:")
print(y_train_balanced.value_counts().sort_index())

Balanced training feature shape: (8278, 34)

Balanced training class counts:
Churn Target
0    4139
1    4139
Name: count, dtype: int64


In [41]:
print("Original test feature shape:", X_test_encoded.shape)
print("Original test target shape:", y_test.shape)

print("\nTest class counts:")
print(y_test.value_counts().sort_index())


Original test feature shape: (1409, 34)
Original test target shape: (1409,)

Test class counts:
Churn Target
0    1035
1     374
Name: count, dtype: int64


In [42]:
print("Balanced training class counts:")
print(y_train_balanced.value_counts().sort_index())

print("\nBalanced training shape:")
print(X_train_balanced.shape)

Balanced training class counts:
Churn Target
0    4139
1    4139
Name: count, dtype: int64

Balanced training shape:
(8278, 34)


## 8. Feature Selection

In [43]:
from sklearn.feature_selection import mutual_info_classif

feature_scores = mutual_info_classif(
    X_train_balanced,
    y_train_balanced,
    random_state=42
)

feature_ranking = pd.DataFrame({
    "Feature": X_train_balanced.columns,
    "Mutual Information Score": feature_scores
})

feature_ranking = feature_ranking.sort_values(
    by="Mutual Information Score",
    ascending=False
).reset_index(drop=True)

feature_ranking

,Feature,Mutual Information Score
0,Total Charges,0.274376
1,Total Revenue,0.270329
2,Total Long Distance Charges,0.254012
3,Avg Monthly Long Distance Charges,0.190679
4,Zip Code,0.183021
5,Latitude,0.175446
6,Longitude,0.170058
7,Contract,0.162496
8,Monthly Charge,0.161989
9,City,0.123772


In [44]:
number_of_selected_features = 20

selected_features = (
    feature_ranking
    .head(number_of_selected_features)["Feature"]
    .tolist()
)

print("Selected features:")
for number, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{number}. {feature}")

Selected features:
1. Total Charges
2. Total Revenue
3. Total Long Distance Charges
4. Avg Monthly Long Distance Charges
5. Zip Code
6. Latitude
7. Longitude
8. Contract
9. Monthly Charge
10. City
11. Number of Referrals
12. Tenure in Months
13. Premium Tech Support
14. Online Security
15. Online Backup
16. Internet Type
17. Number of Dependents
18. Device Protection Plan
19. Streaming Music
20. Avg Monthly GB Download


In [45]:
X_train_selected = X_train_balanced[
    selected_features
].copy()

X_test_selected = X_test_encoded[
    selected_features
].copy()

print(
    "Selected training shape:",
    X_train_selected.shape
)

print(
    "Selected testing shape:",
    X_test_selected.shape
)

Selected training shape: (8278, 20)
Selected testing shape: (1409, 20)


### Feature-Selection Method

Mutual-information scoring was applied using the balanced training dataset only. The features were ranked according to their statistical relationship with the churn target, and the 20 highest-ranked predictors were retained as a candidate feature set for the improved LCS pipeline. The testing dataset was not used to rank or select features, preventing test-set leakage.

The selected feature subset will later be compared with the complete feature set to determine whether the reduction improves LCS performance or model quality.


## 9. Export Preprocessed Dataset

In [46]:
from pathlib import Path

processed_output_directory = Path(
    "../data/feature_engineered"
)

processed_output_directory.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Output directory:",
    processed_output_directory.resolve()
)




Output directory: C:\Users\ACER\Documents\ENGE707-2026-S2-Low-Battery\data\feature_engineered


In [47]:
training_export = X_train_selected.copy()

training_export["Churn Target"] = (
    y_train_balanced.to_numpy()
)

training_export.to_csv(
    processed_output_directory
    / "telecom_train_feature_selected.csv",
    index=False
)

print(
    "Training dataset exported:",
    training_export.shape
)

Training dataset exported: (8278, 21)


In [48]:
testing_export = X_test_selected.copy()

testing_export["Churn Target"] = (
    y_test.to_numpy()
)

testing_export.to_csv(
    processed_output_directory
    / "telecom_test_feature_selected.csv",
    index=False
)

print(
    "Testing dataset exported:",
    testing_export.shape
)

Testing dataset exported: (1409, 21)


In [49]:
complete_training_export = X_train_balanced.copy()

complete_training_export["Churn Target"] = (
    y_train_balanced.to_numpy()
)

complete_testing_export = X_test_encoded.copy()

complete_testing_export["Churn Target"] = (
    y_test.to_numpy()
)

complete_training_export.to_csv(
    processed_output_directory
    / "telecom_train_preprocessed_all_features.csv",
    index=False
)

complete_testing_export.to_csv(
    processed_output_directory
    / "telecom_test_preprocessed_all_features.csv",
    index=False
)

print(
    "Complete training dataset:",
    complete_training_export.shape
)

print(
    "Complete testing dataset:",
    complete_testing_export.shape
)

Complete training dataset: (8278, 35)
Complete testing dataset: (1409, 35)


In [50]:
feature_ranking.to_csv(
    processed_output_directory
    / "mutual_information_feature_ranking.csv",
    index=False
)

print("Feature ranking exported.")


Feature ranking exported.


In [51]:
print("Exported preprocessing files:")

for file_path in sorted(
    processed_output_directory.iterdir()
):
    print(file_path.name)

Exported preprocessing files:
mutual_information_feature_ranking.csv
telecom_test_feature_selected.csv
telecom_test_preprocessed_all_features.csv
telecom_train_feature_selected.csv
telecom_train_preprocessed_all_features.csv


In [52]:
print(
    "Selected training missing values:",
    int(training_export.isna().sum().sum())
)

print(
    "Selected testing missing values:",
    int(testing_export.isna().sum().sum())
)

print(
    "Selected training duplicate rows:",
    int(training_export.duplicated().sum())
)

print(
    "Selected testing duplicate rows:",
    int(testing_export.duplicated().sum())
)

print(
    "\nTraining target distribution:"
)

print(
    training_export["Churn Target"]
    .value_counts()
    .sort_index()
)

print(
    "\nTesting target distribution:"
)

print(
    testing_export["Churn Target"]
    .value_counts()
    .sort_index()
)

Selected training missing values: 0
Selected testing missing values: 0
Selected training duplicate rows: 2776
Selected testing duplicate rows: 0

Training target distribution:
Churn Target
0    4139
1    4139
Name: count, dtype: int64

Testing target distribution:
Churn Target
0    1035
1     374
Name: count, dtype: int64
